In [ ]:
from pathlib import Path
import polars as pl

# =============================
# Config
# =============================
CATEGORY = "Pet_Supplies"
DATA_DIR = Path.cwd() / "prepared"

ITEMS_PATH = DATA_DIR / f"{CATEGORY}_items.parquet"
CLEAN_TITLE_AND_DESC_PATH = DATA_DIR / f"{CATEGORY}_items_cleaned.parquet"
PET_META_PATH = DATA_DIR / f"{CATEGORY}_items_with_metadata.parquet"   # или _metadata_extracted.parquet — подставь свой
OUT_PATH = DATA_DIR / f"{CATEGORY}_items_merged.parquet"

In [ ]:
def normalize_text_cols(df: pl.DataFrame, cols: list[str]) -> pl.DataFrame:
    """
    чистит хвостовые пробелы, приводит пустые строки к null
    """
    exprs = []
    for c in cols:
        if c in df.columns:
            exprs.append(
                pl.when(pl.col(c).is_null())
                .then(None)
                .otherwise(pl.col(c).cast(pl.Utf8).str.replace_all(r"\s+", " ").str.strip_chars())
                .alias(c)
            )
    return df.with_columns(exprs) if exprs else df


def list_to_csv(col: pl.Expr) -> pl.Expr:
    """
    list[str] -> 'a, b, c' ; null -> 'N/A'
    """
    return (
        pl.when(col.is_null())
        .then(pl.lit("N/A"))
        .otherwise(col.list.eval(pl.element().cast(pl.Utf8)).list.join(", "))
    )


In [ ]:
# =============================
# Load
# =============================
print(f"CATEGORY: {CATEGORY}")
print(f"DATA_DIR:  {DATA_DIR}")

items = pl.read_parquet(ITEMS_PATH)
print(f"Items loaded: {items.height:,}  ({ITEMS_PATH.name})")

clean_title_and_desc = pl.read_parquet(CLEAN_TITLE_AND_DESC_PATH)
print(f"Clean title+desc(+features) loaded: {clean_title_and_desc.height:,}  ({CLEAN_TITLE_AND_DESC_PATH.name})")

pet_meta = pl.read_parquet(PET_META_PATH)
print(f"Pet metadata loaded: {pet_meta.height:,}  ({PET_META_PATH.name})")


# =============================
# Prepare clean df (dedup + NA->null + only needed cols)
# =============================
# ожидаем минимум: parent_asin, clean_title, clean_description, clean_features (может отсутствовать)
clean_title_and_desc = clean_title_and_desc.unique(subset=["parent_asin"], keep="last")

# ---- normalize clean_title / clean_description
if "clean_title" not in clean_title_and_desc.columns:
    clean_title_and_desc = clean_title_and_desc.with_columns(pl.lit(None).alias("clean_title"))
if "clean_description" not in clean_title_and_desc.columns:
    clean_title_and_desc = clean_title_and_desc.with_columns(pl.lit(None).alias("clean_description"))

clean_title_and_desc = clean_title_and_desc.with_columns([
    pl.when(
        pl.col("clean_title").is_null()
        | (pl.col("clean_title").cast(pl.Utf8).str.strip_chars() == "")
        | (pl.col("clean_title") == "NA")
    ).then(None).otherwise(pl.col("clean_title")).alias("clean_title"),

    pl.when(
        pl.col("clean_description").is_null()
        | (pl.col("clean_description").cast(pl.Utf8).str.strip_chars() == "")
        | (pl.col("clean_description") == "NA")
    ).then(None).otherwise(pl.col("clean_description")).alias("clean_description"),
])

# ---- normalize clean_features robustly (NO map_elements)
if "clean_features" not in clean_title_and_desc.columns:
    clean_title_and_desc = clean_title_and_desc.with_columns(pl.lit(None).alias("clean_features"))

cf_dtype = clean_title_and_desc.schema.get("clean_features")

# Case 1: already List[...] -> cast inner to Utf8 if needed
if cf_dtype is not None and str(cf_dtype).startswith("List"):
    clean_title_and_desc = clean_title_and_desc.with_columns(
        pl.col("clean_features").cast(pl.List(pl.Utf8), strict=False).alias("clean_features")
    )
else:
    # Case 2: string/other -> normalize to List[Utf8] with single element
    clean_title_and_desc = clean_title_and_desc.with_columns(
        pl.when(
            pl.col("clean_features").is_null()
            | (pl.col("clean_features").cast(pl.Utf8).str.strip_chars() == "")
            | (pl.col("clean_features") == "NA")
        )
        .then(None)
        .otherwise(
            pl.concat_list([pl.col("clean_features").cast(pl.Utf8)])
        )
        .alias("clean_features")
    )

# empty list -> null (удобно для подсчёта и downstream логики)
clean_title_and_desc = clean_title_and_desc.with_columns(
    pl.when(pl.col("clean_features").is_not_null() & (pl.col("clean_features").list.len() == 0))
    .then(None)
    .otherwise(pl.col("clean_features"))
    .alias("clean_features")
)

clean_title_and_desc = clean_title_and_desc.select(
    ["parent_asin", "clean_title", "clean_description", "clean_features"]
)

print(
    "Clean non-null:",
    clean_title_and_desc.select([
        pl.col("clean_title").is_not_null().sum().alias("title_ok"),
        pl.col("clean_description").is_not_null().sum().alias("desc_ok"),
        pl.col("clean_features").is_not_null().sum().alias("features_ok"),
    ]).to_dicts()[0]
)


# =============================
# Prepare pet_meta (dedup + keep cols)
# =============================
pet_meta = pet_meta.unique(subset=["parent_asin"], keep="last")
pet_keep = [
    c for c in [
        "parent_asin",
        "product_type", "pet_species", "life_stage", "primary_use_case",
        "material_or_active", "error",
    ]
    if c in pet_meta.columns
]
pet_meta = pet_meta.select(pet_keep)


# =============================
# Merge
# =============================
# 1) inner join по clean_* — чтобы не тащить карточки без текста под эмбеддинги
df = items.join(clean_title_and_desc, on="parent_asin", how="inner")

# 2) pet metadata — left join
df = df.join(pet_meta, on="parent_asin", how="left")

print(f"Merged rows: {df.height:,}")


# =============================
# Rename original/clean
# =============================
rename_map = {}
if "title" in df.columns:
    rename_map["title"] = "original_title"
if "description_text" in df.columns:
    rename_map["description_text"] = "original_description"
if "features_text" in df.columns:
    rename_map["features_text"] = "original_features_text"
if rename_map:
    df = df.rename(rename_map)

# clean title/desc -> рабочие поля
df = df.rename({
    "clean_title": "title",
    "clean_description": "description_text",
})

# clean_features (List[str]) -> features_text (строка)
if "clean_features" in df.columns:
    df = df.with_columns(
        pl.when(pl.col("clean_features").is_null())
        .then(None)
        .otherwise(pl.col("clean_features").list.join(", "))
        .alias("features_text")
    )

# categories_text: если пусто — подставим main_category
if "categories_text" in df.columns and "main_category" in df.columns:
    df = df.with_columns(
        pl.when(pl.col("categories_text").is_null() | (pl.col("categories_text").str.strip_chars() == ""))
        .then(pl.col("main_category"))
        .otherwise(pl.col("categories_text"))
        .alias("categories_text")
    )


# =============================
# Normalize text fields (optional but helps)
# =============================
df = normalize_text_cols(
    df,
    cols=[
        "title", "description_text", "features_text",
        "main_category", "categories_text", "store",
        "product_type", "pet_species", "life_stage", "primary_use_case",
    ],
)


# =============================
# Build item_context (pet-focused)
# =============================
material_expr = (
    list_to_csv(pl.col("material_or_active"))
    if "material_or_active" in df.columns and str(df.schema["material_or_active"]).startswith("List")
    else pl.col("material_or_active").cast(pl.Utf8).fill_null("N/A")
    if "material_or_active" in df.columns
    else pl.lit("N/A")
)

df = df.with_columns(
    pl.concat_str([
        pl.lit("Title: "), pl.col("title").fill_null(""),
        pl.lit("\nDescription: "), pl.col("description_text").fill_null(""),
        pl.lit("\nCategory: "), pl.col("categories_text").fill_null(pl.col("main_category").fill_null("")),
        pl.lit("\nFeatures: "), pl.col("features_text").fill_null(""),
        pl.lit("\nPet Species: "), pl.col("pet_species").fill_null("N/A"),
        pl.lit("\nProduct Type: "), pl.col("product_type").fill_null("N/A"),
        pl.lit("\nLife Stage: "), pl.col("life_stage").fill_null("N/A"),
        pl.lit("\nPrimary Use Case: "), pl.col("primary_use_case").fill_null("N/A"),
        pl.lit("\nMaterial/Active: "), material_expr,
    ]).alias("item_context")
)


# =============================
# Select final columns
# =============================
FINAL_COLS = [
    "parent_asin",
    "title", "description_text",
    "original_title", "original_description",
    "main_category", "categories_text",
    "features_text", "original_features_text",
    "store",
    "average_rating", "rating_number", "price",
    "product_type", "pet_species", "life_stage", "primary_use_case", "material_or_active", "error",
    "item_context",
    # если хочешь хранить list-форму тоже:
    # "clean_features",
]

existing = [c for c in FINAL_COLS if c in df.columns]
df_out = df.select(existing)

print(f"\nFinal shape: {df_out.shape}  cols={len(existing)}")
print(df_out.head(3))

In [ ]:
# =============================
# Save
# =============================
df_out.write_parquet(OUT_PATH)
print(f"\nSaved to: {OUT_PATH}")

In [ ]:
df_out

In [ ]:
for i in df_out['item_context']:
    print(i)
    print('-'*100)